# E-to-E backbone deterministic and stochastic comparison

This notebook branches from the original E-to-E method comparison without changing it. It keeps a lean deterministic set and adds stochastic path-finding methods that ask a different question: not only "which path maximizes weight?" but also "which edges and termini repeatedly appear under probabilistic flow?"

The deterministic methods are:

1. `greedy_tree_path`
2. `maximum_spanning_tree`
3. `branch_and_bound`
4. `milp_subtour_elimination`
5. `feedback_arc_ordering`

The stochastic methods are:

1. `weighted_random_walk_ensemble`
2. `monte_carlo_tree_search`
3. `boltzmann_path_sampling`

`feedback_arc_ordering` is deterministic but answers a different question from the other four: it maximizes forward-pointing edge weight over a global vertex ordering rather than selecting a single walk, so it scores every positive edge instead of only the edges of one path (Vahidi 2025, arXiv:2506.13799).

All methods use the same E-to-E graph, positive off-diagonal transition weights, and shared output schema. Diagonal/self entries are ignored. Because the methods do not all optimize `summed_weight`, `method_summary.csv` carries an `objective` column and a `comparable_on_summed_weight` flag; read them before ranking.

## Setup

This notebook uses `matrices/mij_EE_matrix.csv`, the same E-to-E input matrix used by the canonical comparison notebook. Outputs are written to `ee_backbone_stochastic_comparison_outputs/` so the original analysis outputs remain untouched.

`milp_subtour_elimination` requires PuLP/CBC. If PuLP is unavailable in the current kernel, install it or skip the MILP cell until the solver environment is ready.

In [ ]:
from pathlib import Path

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

import pandas as pd

from ee_backbone_analysis import (
    EPS,
    best_mst_path_from_seed,
    branch_bound_path,
    build_milp_benchmark_table,
    build_wide_comparison,
    build_tree_adjacency,
    edges_from_path,
    greedy_path,
    maximum_spanning_tree_edges,
    milp_subtour_elimination_path,
    path_to_records,
    prepare_excitatory_matrix_data,
    summarize_methods,
    write_setup_outputs,
)
from ee_backbone_ordering import (
    feedback_arc_ordering,
    ordering_edge_table,
    ordering_path_records,
    permutation_null_forward_fraction,
)
from ee_backbone_stochastic_methods import (
    boltzmann_path_sampling,
    combine_frequency_tables,
    monte_carlo_tree_search_path,
    weighted_random_walk_path_ensemble,
)

INPUT_CSV = Path("matrices/mij_EE_matrix.csv")
OUTDIR = Path("ee_backbone_stochastic_comparison_outputs")

BB_TIME_LIMIT_PER_SEED = 0.25
BB_NODE_LIMIT_PER_SEED = 50_000
MILP_TIME_LIMIT_PER_SEED = 30

RANDOM_WALK_N_WALKS = 2_000
RANDOM_WALK_WEIGHT_POWER = 1.0
RANDOM_WALK_STOP_PROBABILITY = 0.05

MCTS_ITERATIONS = 1_500
MCTS_EXPLORATION_WEIGHT = 1.4
MCTS_ROLLOUT_TEMPERATURE = 0.35

BOLTZMANN_N_SAMPLES = 2_000
BOLTZMANN_TEMPERATURE = 0.25
BOLTZMANN_STOP_PROBABILITY = 0.02

ORDERING_RESTARTS = 50
ORDERING_NULL_DRAWS = 2_000

RANDOM_SEED = 2026


## Data preparation

This section prepares the shared graph. Every row/column label in `mij_EE_matrix.csv` is treated as an excitatory node. Negative weights, zero weights, and diagonal entries are excluded from candidate transitions so every selected edge represents a positive transition between distinct excitatory neurons.

In [ ]:
data = prepare_excitatory_matrix_data(INPUT_CSV, eps=EPS)
write_setup_outputs(data, OUTDIR)

print(f"E-to-E matrix: {data.full_matrix.shape[0]} x {data.full_matrix.shape[1]}")
print(f"Excitatory nodes retained: {len(data.excitatory_nodes)}")
print(f"Positive E-to-E transitions retained: {(data.transition_weights > EPS).sum()}")
print(f"Positive directed E-to-E transition edges retained: {sum(len(v) for v in data.adjacency.values())}")


## Deterministic method 1: Greedy tree/path

The greedy path is the local baseline. Starting from each seed, it repeatedly chooses the largest positive outgoing transition to an unvisited excitatory neuron. It is deterministic, fast, and easy to interpret. Its weakness is also clear: one locally dominant edge can steer the path into a region that prevents a better downstream sequence.

In [ ]:
greedy_results = []
greedy_edges = []

for seed in range(len(data.excitatory_nodes)):
    path = greedy_path(seed, data)
    method = "greedy_tree_path"
    greedy_results.append(path_to_records(seed, method, path, data))
    greedy_edges.extend(edges_from_path(seed, method, path, data))

greedy_df = pd.DataFrame(greedy_results)
greedy_df.head()


## Deterministic method 2: Maximum spanning tree

The maximum spanning tree method first builds an undirected high-weight skeleton by taking the stronger direction for each pair of excitatory neurons. It then extracts the best seed-specific path inside that tree. This method is not a directed path optimum; it is a sparse structural backbone that helps reveal broad corridors of strong pairwise connectivity.

In [ ]:
mst_edges = maximum_spanning_tree_edges(data.transition_weights)
tree_adj = build_tree_adjacency(len(data.excitatory_nodes), mst_edges)

mst_results = []
mst_path_edges = []
for seed in range(len(data.excitatory_nodes)):
    path = best_mst_path_from_seed(seed, tree_adj, data)
    method = "maximum_spanning_tree"
    mst_results.append(path_to_records(seed, method, path, data))
    mst_path_edges.extend(edges_from_path(seed, method, path, data))

mst_df = pd.DataFrame(mst_results)
mst_df.head()


## Deterministic method 3: Branch-and-bound

Branch-and-bound searches directed simple paths while pruning partial paths whose optimistic upper bound cannot beat the current best solution. When it finishes completely it is exact. With time or node-count guardrails, as used here, it should be read as a strong bounded search rather than a certificate of optimality.

In [ ]:
bb_results = []
bb_edges = []

for seed in range(len(data.excitatory_nodes)):
    path, score, states, status = branch_bound_path(
        seed,
        data,
        time_limit=BB_TIME_LIMIT_PER_SEED,
        node_limit=BB_NODE_LIMIT_PER_SEED,
    )
    method = "branch_and_bound"
    bb_results.append(
        path_to_records(
            seed,
            method,
            path,
            data,
            {
                "bb_states_seen": states,
                "bb_status": status,
                "bb_score": score,
            },
        )
    )
    bb_edges.extend(edges_from_path(seed, method, path, data))

bb_df = pd.DataFrame(bb_results)
bb_df.head()


## Deterministic method 4: MILP with subtour elimination

The MILP directly models the maximum-weight simple directed path from each seed. Binary edge and node variables decide which transitions and nodes enter the path; order variables eliminate subtours. The solver status needs care: `pulp.LpStatus[problem.status]` reports `Optimal` both when CBC proves optimality and when it stops on the time limit with an incumbent, so `milp_proven_optimal` (read from `problem.sol_status`) is the field to trust. Only seeds where it is True are certified benchmarks.

In [ ]:
milp_results = []
milp_edges = []

try:
    for seed in range(len(data.excitatory_nodes)):
        (
            path,
            score,
            status,
            objective,
            proven_optimal,
        ) = milp_subtour_elimination_path(
            seed,
            data,
            time_limit=MILP_TIME_LIMIT_PER_SEED,
        )
        method = "milp_subtour_elimination"
        milp_results.append(
            path_to_records(
                seed,
                method,
                path,
                data,
                {
                    "milp_status": status,
                    "milp_score": score,
                    "milp_objective": objective,
                    "milp_time_limit_seconds": MILP_TIME_LIMIT_PER_SEED,
                    "milp_proven_optimal": proven_optimal,
                },
            )
        )
        milp_edges.extend(edges_from_path(seed, method, path, data))
except ImportError as exc:
    print(f"Skipping milp_subtour_elimination because PuLP is unavailable: {exc}")

milp_df = pd.DataFrame(milp_results)
if not milp_df.empty:
    n_proven = int(milp_df['milp_proven_optimal'].sum())
    print(f"MILP: {n_proven} of {len(milp_df)} seeds PROVEN optimal.")
milp_df.head()


## Deterministic method 5: Feedback-arc ordering

The four methods above each select one walk. This one selects an ordering of all 32 excitatory types that maximizes the total weight of forward-pointing edges, i.e. minimizes the weight of the feedback arc set. It therefore scores every positive transition against the ordering rather than only the transitions belonging to one path.

The search follows Vahidi (2025): a weighted Eades-Lin-Smyth greedy construction, gain-aware local refinement by best-position reinsertion, and exact decomposition over strongly connected components (all edges between distinct SCCs can be made forward at once by ordering the condensation topologically, so feedback weight is only ever incurred within an SCC). Randomized restarts are added because refinement converges to a local optimum of the reinsertion neighbourhood.

Once the ordering is fixed the forward-edge subgraph is acyclic, so the per-seed maximum-weight forward path is exact in `O(V + E)`. Those paths are reported for schema compatibility only: they are shorter and lighter than the free-path methods by construction, because a seed placed late in the ordering has few forward options while the MILP may move backward through the ordering freely. Judge this method by its forward-weight fraction against the permutation null, not by `summed_weight`.

Reference: Vahidi, S. (2025). *Feedforward Ordering in Neural Connectomes via Feedback Arc Minimization.* arXiv:2506.13799.


In [ ]:
ordering = feedback_arc_ordering(data, n_restarts=ORDERING_RESTARTS, random_seed=RANDOM_SEED)
ordering_null = permutation_null_forward_fraction(
    data.transition_weights,
    ordering.metrics["forward_weight_fraction"],
    n_null=ORDERING_NULL_DRAWS,
    random_seed=RANDOM_SEED,
)
ordering_results, ordering_edges = ordering_path_records(ordering, data)
ordering_df = pd.DataFrame(ordering_results)

print(f"SCCs: {len(ordering.scc_sizes)}  sizes: {ordering.scc_sizes}")
print(f"forward-weight fraction: {ordering.metrics['forward_weight_fraction']:.4f}")
print(f"feedback weight: {ordering.metrics['feedback_weight']:,.1f} "
      f"across {ordering.metrics['n_feedback_edges']} of {ordering.metrics['n_edges']} edges")
print(f"permutation null mean: {ordering_null['null_mean_forward_fraction']:.4f} "
      f"(sd {ordering_null['null_sd_forward_fraction']:.4f}), "
      f"empirical p = {ordering_null['empirical_p_forward_fraction']:.4g}")

ordering_df.head()


## Stochastic method 1: Weighted random-walk / diffusion path ensemble

This method samples many simple paths from each seed. At each node, outgoing transitions are chosen with probability proportional to edge weight. This is the closest stochastic complement to connectomics-style diffusion/random-walk communication: rather than selecting only one optimum, it estimates which edges, terminal nodes, and path weights are likely under locally weighted flow.

The reported path is the best sampled path for compatibility with the deterministic table. The more important stochastic diagnostics are the mean sampled score, terminal entropy, number of unique terminals, and edge visit frequencies.

In [ ]:
random_walk_results = []
random_walk_edges = []
random_walk_edge_frequencies = []
random_walk_terminal_distributions = []

for seed in range(len(data.excitatory_nodes)):
    path, diagnostics, edge_frequency, terminal_distribution = weighted_random_walk_path_ensemble(
        seed,
        data,
        n_walks=RANDOM_WALK_N_WALKS,
        weight_power=RANDOM_WALK_WEIGHT_POWER,
        stop_probability=RANDOM_WALK_STOP_PROBABILITY,
        random_seed=RANDOM_SEED,
    )
    method = "weighted_random_walk_ensemble"
    random_walk_results.append(path_to_records(seed, method, path, data, diagnostics))
    random_walk_edges.extend(edges_from_path(seed, method, path, data))
    random_walk_edge_frequencies.append(edge_frequency)
    random_walk_terminal_distributions.append(terminal_distribution)

random_walk_df = pd.DataFrame(random_walk_results)
random_walk_df.head()


## Stochastic method 2: Monte Carlo tree search

Monte Carlo tree search uses repeated simulations to decide which path prefixes deserve more attention. It balances exploitation of high-scoring prefixes with exploration of under-sampled alternatives using an upper-confidence rule. This is useful when the search tree is too large to exhaustively enumerate but you still want a directed search process rather than independent random samples.

In [ ]:
mcts_results = []
mcts_edges = []

for seed in range(len(data.excitatory_nodes)):
    path, diagnostics = monte_carlo_tree_search_path(
        seed,
        data,
        iterations=MCTS_ITERATIONS,
        exploration_weight=MCTS_EXPLORATION_WEIGHT,
        rollout_temperature=MCTS_ROLLOUT_TEMPERATURE,
        random_seed=RANDOM_SEED,
    )
    method = "monte_carlo_tree_search"
    mcts_results.append(path_to_records(seed, method, path, data, diagnostics))
    mcts_edges.extend(edges_from_path(seed, method, path, data))

mcts_df = pd.DataFrame(mcts_results)
mcts_df.head()


## Stochastic method 3: Boltzmann path sampling

Boltzmann path sampling draws paths from a softmax over outgoing edge weights. The temperature controls how deterministic the walk is. At low temperature, the sampler behaves more like greedy selection; at high temperature, it explores more alternatives. This makes it a useful sensitivity analysis: if the same edges dominate across temperatures or samples, the backbone choice is less likely to be a fragile artifact of one deterministic rule.

In [ ]:
boltzmann_results = []
boltzmann_edges = []
boltzmann_edge_frequencies = []
boltzmann_terminal_distributions = []

for seed in range(len(data.excitatory_nodes)):
    path, diagnostics, edge_frequency, terminal_distribution = boltzmann_path_sampling(
        seed,
        data,
        n_samples=BOLTZMANN_N_SAMPLES,
        temperature=BOLTZMANN_TEMPERATURE,
        stop_probability=BOLTZMANN_STOP_PROBABILITY,
        random_seed=RANDOM_SEED,
    )
    method = "boltzmann_path_sampling"
    boltzmann_results.append(path_to_records(seed, method, path, data, diagnostics))
    boltzmann_edges.extend(edges_from_path(seed, method, path, data))
    boltzmann_edge_frequencies.append(edge_frequency)
    boltzmann_terminal_distributions.append(terminal_distribution)

boltzmann_df = pd.DataFrame(boltzmann_results)
boltzmann_df.head()


## Combine and write outputs

This section combines all deterministic and stochastic best-path records into one comparison table. Separate frequency tables preserve the ensemble information for the random-walk and Boltzmann methods.

In [ ]:
OUTDIR.mkdir(exist_ok=True)
comparison = pd.concat(
    [
        greedy_df,
        mst_df,
        bb_df,
        milp_df,
        ordering_df,
        random_walk_df,
        mcts_df,
        boltzmann_df,
    ],
    ignore_index=True,
)
edge_table = pd.DataFrame(
    greedy_edges
    + mst_path_edges
    + bb_edges
    + milp_edges
    + ordering_edges
    + random_walk_edges
    + mcts_edges
    + boltzmann_edges
)

wide = build_wide_comparison(comparison)
method_summary = summarize_methods(comparison)
milp_benchmark = build_milp_benchmark_table(comparison)

comparison.to_csv(OUTDIR / "method_comparison_long.csv", index=False)
wide.to_csv(OUTDIR / "method_comparison_wide.csv", index=False)
edge_table.to_csv(OUTDIR / "all_methods_edges.csv", index=False)
method_summary.to_csv(OUTDIR / "method_summary.csv", index=False)
if not milp_benchmark.empty:
    milp_benchmark.to_csv(OUTDIR / "milp_benchmark_gaps.csv", index=False)

edge_frequency = combine_frequency_tables(
    random_walk_edge_frequencies + boltzmann_edge_frequencies,
    "edge_frequency",
)
terminal_distribution = combine_frequency_tables(
    random_walk_terminal_distributions + boltzmann_terminal_distributions,
    "terminal_distribution",
)

if not edge_frequency.empty:
    edge_frequency.to_csv(OUTDIR / "stochastic_edge_frequencies.csv", index=False)
if not terminal_distribution.empty:
    terminal_distribution.to_csv(OUTDIR / "stochastic_terminal_distributions.csv", index=False)

ordering.position_table().to_csv(OUTDIR / "backbone_ordering.csv", index=False)
pd.Series(ordering.metrics).to_csv(OUTDIR / "ordering_metrics.csv")
pd.Series(ordering_null).to_csv(OUTDIR / "ordering_permutation_null.csv")

print("Wrote comparison outputs to:", OUTDIR)
method_summary


## Stochastic ensemble diagnostics

The stochastic methods should not be judged only by their best sampled path. The ensemble diagnostics below show whether sampled paths concentrate on a small set of edges and termini or spread across many alternatives. High entropy and many unique termini indicate uncertainty or multiple plausible backbones; low entropy and repeated high-probability edges indicate a more stable stochastic choice.

In [ ]:
stochastic_columns = [
    "method",
    "seed",
    "summed_weight",
    "mean_sample_score",
    "median_sample_score",
    "sample_score_std",
    "terminal_entropy_bits",
    "top_edge_visit_probability",
    "n_unique_terminals_sampled",
    "n_unique_edges_sampled",
]
available_columns = [column for column in stochastic_columns if column in comparison.columns]
stochastic_diagnostics = comparison.loc[
    comparison["method"].isin(["weighted_random_walk_ensemble", "boltzmann_path_sampling"]),
    available_columns,
]
stochastic_diagnostics.head(20)


## Plot path-weight distributions

The plot gives a compact score comparison across deterministic best paths and stochastic best sampled paths. For stochastic methods, use the ensemble diagnostics above before making claims; a high best sample with high entropy may be less stable than a slightly lower score with concentrated edge visitation.

In [ ]:
if plt is None:
    print("matplotlib is not installed; skipping boxplot.")
else:
    # feedback_arc_ordering optimizes a global ordering, not a path weight;
    # including it here would misrepresent it. See ordering_metrics.csv.
    plot_df = comparison.loc[
        comparison["method"] != "feedback_arc_ordering", ["method", "summed_weight"]
    ].copy()
    ax = plot_df.boxplot(column="summed_weight", by="method", rot=60)
    plt.suptitle("")
    plt.title("Deterministic and stochastic E-to-E backbone path weights")
    plt.xlabel("Method")
    plt.ylabel("Path weight")
    plt.tight_layout()
    plt.show()


## Suggested significance tests and null models

These are recommended analyses for deciding whether a selected backbone is stronger or more stable than expected by chance. They are not all run by default because each null model answers a slightly different scientific question.

| Approach | What it tests | Suggested statistic |
| --- | --- | --- |
| Weight-shuffled fixed topology null | Whether observed edge weights, not just graph topology, drive the selected backbone | Path weight, gap to null mean, empirical p-value |
| Degree/strength-preserving directed rewiring | Whether the backbone depends on specific source-target pairing beyond degree or strength sequence | Path weight, edge overlap/Jaccard with observed path |
| Row-wise target shuffle | Whether each sender's outgoing weight distribution is enough to explain the backbone | Terminal distribution, path score, edge recurrence |
| Sign-preserving full-matrix null | Whether E-to-E positive structure is special relative to the signed full matrix | E-to-E path score and retained edge count |
| Bootstrap or noise perturbation of weights | Whether selected edges are stable under measurement uncertainty | Edge inclusion frequency and path-rank confidence interval |
| Stochastic ensemble consensus | Whether deterministic selected edges are also high-probability under random-walk/Boltzmann flow | Edge visitation probability, terminal entropy |
| Seed-label permutation | Whether high scores are tied to specific biological seed identities | Seed-specific score percentile under relabeling |

For empirical p-values, use `(1 + number of null scores >= observed score) / (1 + number of null samples)` so the p-value is never exactly zero. For many seeds or many edges, control false discoveries with Benjamini-Hochberg FDR. For edge-level claims, report both statistical significance and effect size, such as observed edge frequency minus null edge frequency.

In [ ]:
null_model_suggestions = pd.DataFrame([
    {
        "null_model": "weight_shuffled_fixed_topology",
        "preserves": "directed graph topology and edge count",
        "randomizes": "weights across existing positive E-to-E edges",
        "primary_readout": "observed path weight percentile versus null",
    },
    {
        "null_model": "degree_strength_preserving_rewire",
        "preserves": "approximate in/out degree and source/target strength",
        "randomizes": "specific source-target pairing",
        "primary_readout": "path weight and edge overlap versus null",
    },
    {
        "null_model": "rowwise_target_shuffle",
        "preserves": "each source neuron's outgoing weight multiset",
        "randomizes": "which excitatory targets receive those weights",
        "primary_readout": "terminal distribution and edge recurrence",
    },
    {
        "null_model": "weight_noise_bootstrap",
        "preserves": "observed topology and approximate weights",
        "randomizes": "weights by measurement-noise perturbation",
        "primary_readout": "edge inclusion confidence and path-rank stability",
    },
    {
        "null_model": "stochastic_ensemble_consensus",
        "preserves": "observed graph and weights",
        "randomizes": "path realization under weighted flow",
        "primary_readout": "edge visit probability and terminal entropy",
    },
])
null_model_suggestions.to_csv(OUTDIR / "null_model_suggestions.csv", index=False)
null_model_suggestions
